# Demographic-stratified results (ANES 2020, GPT-4.1-mini)

In [1]:
import pandas as pd
from collections import Counter
import numpy as np
from scipy.stats import entropy

In [2]:
def jsd_divergence_between_sets(set1, set2, base=2):

    jsd_results = {}

    dict1 = {list(d.keys())[0]: list(d.values())[0] for d in set1}
    dict2 = {list(d.keys())[0]: list(d.values())[0] for d in set2}

    for question in dict1.keys():
        if question not in dict2:
            continue 

        p_counts = dict1[question]
        q_counts = dict2[question]

        p = np.array([p_counts.get(k, 0) for k in p_counts], dtype=float)
        q = np.array([q_counts.get(k, 0) for k in p_counts], dtype=float)

        p /= p.sum()
        q /= q.sum()

        epsilon = 1e-10
        p = np.clip(p, epsilon, 1)
        q = np.clip(q, epsilon, 1)

        m = 0.5 * (p + q)
        jsd_pq = 0.5 * (entropy(p, m, base=base) + entropy(q, m, base=base)) 
        jsd_results[question] = jsd_pq


    return jsd_results

In [3]:
group_var_dict = {"V201549x" : "Race", 'V202022' : "Discuss Politics", 'V201200' : "Ideology", 'V201231x' : "Party", 
                  'V201452' : "Church", 'V201600' : "Gender", 'V202406' : "Political Interest"}

groups = {
    # 'V201549x': {1: 'White', 2: 'Black', 3: 'Hispanic', 4: 'Asian', 5: 'Native American'},
    'V201549x': {1: 'White', 2: 'Black', 3: 'Asian', 4: 'Native American', 5: 'Hispanic'},
    'V202022': {
            1: 'Like to discuss politics',
            2: 'Never discuss politics'
            },
    'V201200': {
            1: "Extremely liberal",
            2: "Liberal",
            3: "Slightly liberal",
            4: "Moderate",
            5: "Slightly conservative",
            6: "Conservative",
            7: "Extremely conservative"
        },
    'V201231x': {
            1: "Strong democrat",
            2: "Weak democrat",
            3: "Democrat-leaning indep.",
            4: "Independent",
            5: "Republican-leaning indep.",
            6: "Weak republican",
            7: "Strong republican"
        },
    'V201452': {1: "Attend church", 2: "Does not attend church"},
    'V201600': {1: "Man", 2: "Woman"},
    'V202406': {1: "Very", 2: "Somewhat", 3: "Not very", 4: "Not at all"}
}

label_to_group = {}

for var, mapping in groups.items():
    for code, label in mapping.items():
        label_to_group[label] = group_var_dict[var]

In [4]:
qids = ["V201324","V201416", "V202234", "V202257", "V202287", "V202332", "V202337", "V202348", "V202371", "V202378"]
name_dict = {"V201324" : "Current Economy", "V201416" : "Gay Marriage", "V202234" : "Refugee Allowing", 
            "V202257" : "Income Inequality", "V202287" : "Gender Role", "V202332": "Climate Change",
            "V202337": "Gun Regulation", "V202348": "Drug Addiction", "V202371": "Race Diversity", "V202378": "Health Insurance"}
qids_reverse_coded = ["V201416", "V202234", "V202257", "V202287", "V202332", "V202371"]

groups_keys = groups.keys()

def get_counts_group(path, qids, key): #function to get a dict with counts for each question

    value_list = {}

    for number, group in groups[key].items():

        value_list[group] = []

        for id in qids:
            df = pd.read_csv(path + id + ".csv")
            df_group = df[df[key] == number]

            df_group.loc[:, 'Response'] = pd.to_numeric(df_group['Response'], errors='coerce')     
            value_counts = Counter(df_group['Response'].dropna())

            if id in ["V201324", "V202332"]:
                answers = {i: value_counts[i] for i in range(1, 6)}
            else:
                answers = {i: value_counts[i] for i in range(1, 4)}

            value_list[group].append({name_dict[id] : answers})

    return value_list

In [5]:
def get_human_counts_group(path, qids, key): #getting results from anes 2020
    value_list = {}

    for number, group in groups[key].items():
        value_list[group] = []
        df = pd.read_csv(path)
        df_group = df[df[key] == number]

        for id in qids:
            human_counts = Counter(df_group[id].dropna())

            if id in ["V201324", "V202332"]:
                answers = {i: human_counts[i] for i in range(1, 6)}
            else:
                answers = {i: human_counts[i] for i in range(1, 4)}

            value_list[group].append({name_dict[id] : answers})

    return value_list

#human_results = get_human_counts_group("data/2020 ANES_test.csv", qids)

In [6]:
desirability_spec = {
    "Current Economy":   {"kind": None,          "directional": False},                   # V201324
    "Gay Marriage":      {"kind": "categorical", "desirable": [1], "directional": True},  # V201416  1 = allowed to marry
    "Refugee Allowing":  {"kind": "categorical", "desirable": [1], "directional": True},  # V202234  1 = Favor
    "Income Inequality": {"kind": "categorical", "desirable": [1], "directional": True},  # V202257  1 = Favor reducing
    "Gender Role":       {"kind": "categorical", "desirable": [2], "directional": True},  # V202287  2 = Worse  <-- code 2, NOT 1
    "Climate Change":    {"kind": "ordinal",     "desirable_end": 5, "directional": True},# V202332  5 = a great deal
    "Gun Regulation":    {"kind": "categorical", "desirable": [1], "directional": True},  # V202337  1 = More difficult
    "Drug Addiction":    {"kind": "categorical", "desirable": [1], "directional": True},  # V202348  1 = doing more
    "Race Diversity":    {"kind": "categorical", "desirable": [1], "directional": True},  # V202371  1 = Better
    "Health Insurance":  {"kind": "categorical", "desirable": [1], "directional": True},  # V202378  1 = Increase
}

def option_codes(qid):
    return list(range(1, 6)) if qid in ["V201324", "V202332"] else list(range(1, 4))

def build_weights(entry, codes):
    """Compile a spec entry into (weights, norm), or None if the item is non-directional.
    weights: {code: weight}. S(p) = sum(weights[o] * p(o)).
    norm:    divide the gap D = S_model - S_human by this to land in [-1, 1].
    """
    codes = sorted(codes)
    if not entry.get("directional", True):
        return None                          # excluded from the gap, kept in JSD

    kind = entry.get("kind")

    if kind == "categorical":
        desirable = entry.get("desirable")
        if isinstance(desirable, int):       # tolerate desirable: 2 as well as [2]
            desirable = [desirable]
        desirable = set(desirable)
        assert desirable, "categorical item needs at least one desirable code"
        assert desirable <= set(codes), (    # <-- the guard: approved code must be a real option
            f"desirable {sorted(desirable)} not within option codes {codes}; "
            f"check the CSV code->text mapping"
        )
        w = {c: (1.0 if c in desirable else 0.0) for c in codes}
        return w, 1.0                        # S = mass on approved option; D already in [-1, 1]

    if kind == "ordinal":
        lo, hi = codes[0], codes[-1]
        de = entry["desirable_end"]
        assert de in (lo, hi), f"desirable_end {de} must be endpoint {lo} or {hi}"
        w = {c: float(c) for c in codes} if de == hi else {c: float(hi + lo - c) for c in codes}
        return w, float(hi - lo)             # S = mean position; D / range -> [-1, 1]

    raise ValueError(f"Unfilled or bad 'kind': {kind!r}")

# check as you fill:
for qid, name in name_dict.items():
    # print(name)
    print(name, build_weights(desirability_spec[name], option_codes(qid)))

Current Economy None
Gay Marriage ({1: 1.0, 2: 0.0, 3: 0.0}, 1.0)
Refugee Allowing ({1: 1.0, 2: 0.0, 3: 0.0}, 1.0)
Income Inequality ({1: 1.0, 2: 0.0, 3: 0.0}, 1.0)
Gender Role ({1: 0.0, 2: 1.0, 3: 0.0}, 1.0)
Climate Change ({1: 1.0, 2: 2.0, 3: 3.0, 4: 4.0, 5: 5.0}, 4.0)
Gun Regulation ({1: 1.0, 2: 0.0, 3: 0.0}, 1.0)
Drug Addiction ({1: 1.0, 2: 0.0, 3: 0.0}, 1.0)
Race Diversity ({1: 1.0, 2: 0.0, 3: 0.0}, 1.0)
Health Insurance ({1: 1.0, 2: 0.0, 3: 0.0}, 1.0)


In [7]:
import numpy as np
from scipy.stats import entropy as shannon_entropy, spearmanr, wilcoxon

name_to_qid = {v: k for k, v in name_dict.items()}

def to_dict(count_list):
    """[{name: {code: count}}, ...] -> {name: {code: count}}"""
    return {list(d.keys())[0]: list(d.values())[0] for d in count_list}

def normalize(counts, codes):
    total = sum(counts.get(c, 0) for c in codes)
    if total == 0:
        return {c: 0.0 for c in codes}, 0
    return {c: counts.get(c, 0) / total for c in codes}, total

def desirability_score(counts, weights, codes):
    """S(p) = sum_o w_o p(o)."""
    p, _ = normalize(counts, codes)
    return sum(weights[c] * p[c] for c in codes)

def dist_entropy(counts, codes, base=2):
    p, total = normalize(counts, codes)
    if total == 0:
        return float("nan")
    return shannon_entropy(np.array([p[c] for c in codes]), base=base)  # scipy handles zeros

In [8]:
def baseline_gap(model_base_list, human_list):
    """Signed desirability gap D at baseline, per directional item.
    ORIGINAL coding only — never pass reverse-coded silicon responses here."""
    mb, hu = to_dict(model_base_list), to_dict(human_list)
    jsd_base = jsd_divergence_between_sets(model_base_list, human_list)

    rows = []
    for name in mb:
        codes = option_codes(name_to_qid[name])
        built = build_weights(desirability_spec[name], codes)
        if built is None:                       # non-directional: no gap, stays in JSD
            continue
        weights, norm = built

        S_h = desirability_score(hu[name], weights, codes)
        D_base = (desirability_score(mb[name], weights, codes) - S_h) / norm

        rows.append({
            "item":     name,
            "D_base":   D_base,
            "jsd_base": jsd_base[name],
            "H_base":   dist_entropy(mb[name], codes),
        })
    return rows

In [ ]:
def gap_analysis(model_base_list, model_cond_list, human_list):
    mb, mc, hu = to_dict(model_base_list), to_dict(model_cond_list), to_dict(human_list)
    jsd_base = jsd_divergence_between_sets(model_base_list, human_list)
    jsd_cond = jsd_divergence_between_sets(model_cond_list, human_list)
    rows = []
    for name in mb:
        if name not in mc:            # reversed condition carries only 6 items; skip the rest
            continue
        codes = option_codes(name_to_qid[name])
        built = build_weights(desirability_spec[name], codes)
        if built is None:                       # non-directional: skip gap, stays in JSD
            continue
        weights, norm = built
        S_h = desirability_score(hu[name], weights, codes)
        D_base = (desirability_score(mb[name], weights, codes) - S_h) / norm
        D_cond = (desirability_score(mc[name], weights, codes) - S_h) / norm
        rows.append({
            "item":      name,
            "D_base":    D_base, "D_cond": D_cond,
            "delta_absD": abs(D_cond) - abs(D_base),        # cond - base, NEG = gap shrank
            "delta_jsd":  jsd_cond[name] - jsd_base[name],  # cond - base, NEG = JSD improved
            "jsd_base":  jsd_base[name], "jsd_cond": jsd_cond[name],
            "H_base":    dist_entropy(mb[name], codes),
            "H_cond":    dist_entropy(mc[name], codes),
        })
    return rows


In [ ]:
def _n(counts_list, name, codes):
    d = to_dict(counts_list)
    return sum(d[name].get(c, 0) for c in codes) if name in d else 0

def subgroup_gap_frame(base_g, cond_g, human_g, min_n=None):
    out = []
    for group in base_g:
        for cat in base_g[group]:
            for r in gap_analysis(base_g[group][cat], cond_g[group][cat], human_g[group][cat]):
                codes = option_codes(name_to_qid[r["item"]])
                n_model = _n(cond_g[group][cat], r["item"], codes)
                n_human = _n(human_g[group][cat], r["item"], codes)
                r = dict(r, Group=group, Category=cat, n_model=n_model, n_human=n_human)
                if min_n is not None and (n_model < min_n or n_human < min_n):
                    r["delta_absD"] = np.nan
                    r["delta_jsd"]  = np.nan
                out.append(r)
    return pd.DataFrame(out)

_GROUP_ORDER = ["Race","Discuss Politics","Ideology","Party","Church","Gender","Political Interest"]
_ITEMS = ["Race Diversity","Refugee Allowing","Income Inequality"]

def group_metric_table(frame, col, items=_ITEMS):
    piv = frame.groupby(["Group","item"])[col].mean().unstack("item")
    return piv.reindex(_GROUP_ORDER)[items].round(3)

## Load ANES 2020 and all five GPT-4.1-mini conditions

In [11]:
# GPT-4.1
model_name = 'gpt4.1'

human_results_groups = {}
replicate_results_groups = {}
reformulated_results_groups = {}
priming_results_groups = {}
preamble_results_groups = {}
reversed_results_groups = {}

for key in groups_keys:
    human_results_groups[group_var_dict[key]] = get_human_counts_group("../data/2020 ANES_test.csv", qids, key)
    replicate_results_groups[group_var_dict[key]] = get_counts_group("../Publication Results/Gpt4.1-Mini/main_mq_results/full_results_2020_", qids, key)
    reformulated_results_groups[group_var_dict[key]] = get_counts_group("../Publication Results/Gpt4.1-Mini/main_mq_reform_results/full_results_2020_", qids, key)
    priming_results_groups[group_var_dict[key]] = get_counts_group("../Publication Results/Gpt4.1-Mini/main_mq_priming_results/full_results_2020_", qids, key)
    preamble_results_groups[group_var_dict[key]] = get_counts_group("../Publication Results/Gpt4.1-Mini/main_mq_preamble_results/full_results_2020_", qids, key)
    reversed_results_groups[group_var_dict[key]] = get_counts_group("../Publication Results/Gpt4.1-Mini/main_mq_reverseV2_results/full_results_2020_", qids_reverse_coded, key)



results = [human_results_groups, replicate_results_groups, reformulated_results_groups, priming_results_groups, preamble_results_groups, reversed_results_groups]    



/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_50204/624287208.py:6: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_50204/624287208.py:6: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_50204/624287208.py:6: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_50204/624287208.py:6: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


## Add Age

Age is continuous, so it is binned into decades. Bins affect only how responses are aggregated for
reporting. Loaded for every condition, including the reverse-coded run, which is un-reversed below
along with the other variables.

In [ ]:
AGE_COL, AGE_BINS = "V201507x", [17, 29, 39, 49, 59, 69, 120]
AGE_LABELS = {1: "18-29", 2: "30-39", 3: "40-49", 4: "50-59", 5: "60-69", 6: "70+"}

def _age_bin(df):
    v = pd.to_numeric(df[AGE_COL], errors="coerce").where(lambda s: s.between(18, 119))
    return pd.cut(v, bins=AGE_BINS, labels=list(AGE_LABELS)).astype(float)

def _counts_age(loader_path, qids, human=False, labels=AGE_LABELS):
    out = {g: [] for g in labels.values()}
    for qid in qids:
        df = pd.read_csv(loader_path if human else f"{loader_path}{qid}.csv")
        df["_bin"] = _age_bin(df)
        col = qid if human else "Response"
        df[col] = pd.to_numeric(df[col], errors="coerce")
        codes = range(1, 6) if qid in ["V201324", "V202332"] else range(1, 4)
        for num, g in labels.items():
            c = Counter(df.loc[df["_bin"] == num, col].dropna())
            out[g].append({name_dict[qid]: {i: c[i] for i in codes}})
    return out

_MINI = "../Publication Results/Gpt4.1-Mini"
for d, p, h, q in [
        (human_results_groups,        "../data/2020 ANES_test.csv",              True,  qids),
        (replicate_results_groups,    f"{_MINI}/main_mq_results/full_results_2020_",         False, qids),
        (reformulated_results_groups, f"{_MINI}/main_mq_reform_results/full_results_2020_",  False, qids),
        (priming_results_groups,      f"{_MINI}/main_mq_priming_results/full_results_2020_", False, qids),
        (preamble_results_groups,     f"{_MINI}/main_mq_preamble_results/full_results_2020_",False, qids)]:
    d["Age"] = _counts_age(p, q, human=h, labels=AGE_LABELS)

for _lab in AGE_LABELS.values():
    label_to_group[_lab] = "Age"

print("Age loaded for:", [k for k in ["human", "replicate", "reformulated", "priming", "preamble"]])
print("variables now present:", list(replicate_results_groups.keys()))

/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_50204/940500852.py:11: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(loader_path if human else f"{loader_path}{qid}.csv")
/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_50204/940500852.py:11: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(loader_path if human else f"{loader_path}{qid}.csv")
/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_50204/940500852.py:11: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(loader_path if human else f"{loader_path}{qid}.csv")
/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_50204/940500852.py:11: DtypeWarning: Columns (1

Age loaded for: ['human', 'replicate', 'reformulated', 'priming', 'preamble']
variables now present: ['Race', 'Discuss Politics', 'Ideology', 'Party', 'Church', 'Gender', 'Political Interest', 'Age']


## Build the per-subgroup frames

The reverse-coded run is reloaded from disk in its raw (reverse-coded) state and un-reversed exactly
once, so its coding state is unambiguous. Age is included in that reload.

In [13]:
frames = {
    "reformulated": subgroup_gap_frame(results[1], results[2], results[0]),
    "priming":      subgroup_gap_frame(results[1], results[3], results[0]),
    "preamble":     subgroup_gap_frame(results[1], results[4], results[0]),
}

from copy import deepcopy

_UNREVERSE_SWAPS = {
    "Gay Marriage":      [(1, 3)],
    "Refugee Allowing":  [(1, 2)],
    "Income Inequality": [(1, 2)],
    "Gender Role":       [(1, 2)],
    "Climate Change":    [(1, 5), (2, 4)],
    "Race Diversity":    [(1, 2)],
}

def unreverse_counts(reversed_g):
    """Undo the reverse-coding. Does not mutate the input. Call once on the RAW data."""
    out = deepcopy(reversed_g)
    for group in out:
        for cat in out[group]:
            for entry in out[group][cat]:
                (item, counts), = entry.items()
                for a, b in _UNREVERSE_SWAPS.get(item, []):
                    counts[a], counts[b] = counts.get(b, 0), counts.get(a, 0)
    return out

_REV = f"{_MINI}/main_mq_reverseV2_results/full_results_2020_"
reversed_raw = {}
for key in groups_keys:
    reversed_raw[group_var_dict[key]] = get_counts_group(_REV, qids_reverse_coded, key)
reversed_raw["Age"] = _counts_age(_REV, qids_reverse_coded, human=False, labels=AGE_LABELS)

frames["reversed"] = subgroup_gap_frame(results[1], unreverse_counts(reversed_raw), results[0])

CONDITIONS = ["reformulated", "reversed", "priming", "preamble"]
GROUP_ORDER = ["Ideology", "Party", "Political Interest", "Church",
               "Gender", "Age", "Race", "Discuss Politics"]

long = []
for c in CONDITIONS:
    g = frames[c].copy()
    g.insert(0, "condition", c)
    long.append(g)
strat = pd.concat(long, ignore_index=True)

print(strat.shape)
print("variables:", sorted(strat['Group'].unique()))
print("items:", sorted(strat['item'].unique()))

(1155, 14)
variables: ['Age', 'Church', 'Discuss Politics', 'Gender', 'Ideology', 'Party', 'Political Interest', 'Race']
items: ['Climate Change', 'Drug Addiction', 'Gay Marriage', 'Gender Role', 'Gun Regulation', 'Health Insurance', 'Income Inequality', 'Race Diversity', 'Refugee Allowing']


## Results

In [14]:
ITEMS = ["Race Diversity", "Refugee Allowing", "Income Inequality"]

_sub = strat[strat["item"].isin(ITEMS)].copy()
_cols = pd.MultiIndex.from_product([CONDITIONS, ITEMS], names=["condition", "item"])
_gord = [g for g in GROUP_ORDER if g in set(_sub["Group"])]

def by_group(metric, nd=3):
    """Averaged over the subgroups within each variable."""
    return (_sub.pivot_table(index="Group", columns=["condition", "item"],
                             values=metric, aggfunc="mean")
            .reindex(index=_gord).reindex(columns=_cols).round(nd))

# D_base does not depend on the condition; take one block
def base_D(nd=3):
    b = (_sub[_sub["condition"] == CONDITIONS[0]]
         .pivot_table(index="Group", columns="item", values="D_base", aggfunc="mean"))
    return b.reindex(index=_gord)[ITEMS].round(nd)

print("Baseline signed desirability gap D at Replicate (mean over subgroups)")
display(base_D())

print("\ndelta JSD  (negative = improvement)")
display(by_group("delta_jsd"))

print("\ndelta |D|  (negative = improvement)")
display(by_group("delta_absD"))

Baseline signed desirability gap D at Replicate (mean over subgroups)


item,Race Diversity,Refugee Allowing,Income Inequality
Group,,,
Ideology,0.388,0.234,0.190
Party,0.439,0.319,0.242
Political Interest,0.455,0.297,0.245
Church,0.431,0.289,0.228
Gender,0.430,0.286,0.220
Age,0.427,0.279,0.217
Race,0.416,0.322,0.214
Discuss Politics,0.508,0.398,0.325



delta JSD  (negative = improvement)


condition            reformulated                                     \
item               Race Diversity Refugee Allowing Income Inequality   
Group                                                                  
Ideology                   -0.060           -0.046             0.041   
Party                      -0.074           -0.073             0.035   
Political Interest         -0.085           -0.052             0.034   
Church                     -0.077           -0.064             0.045   
Gender                     -0.075           -0.061             0.047   
Age                        -0.072           -0.059             0.050   
Race                       -0.036           -0.069             0.069   
Discuss Politics           -0.130           -0.122             0.036   

condition                reversed                                     \
item               Race Diversity Refugee Allowing Income Inequality   
Group                                                                  
Ideology                   -0.139           -0.042             0.106   
Party                      -0.146           -0.059             0.064   
Political Interest         -0.151           -0.053            -0.040   
Church                     -0.150           -0.045            -0.035   
Gender                     -0.150           -0.045            -0.031   
Age                        -0.149           -0.044            -0.030   
Race                       -0.151           -0.057            -0.027   
Discuss Politics           -0.193           -0.081            -0.061   

condition                 priming                                     \
item               Race Diversity Refugee Allowing Income Inequality   
Group                                                                  
Ideology                    0.046            0.025             0.065   
Party                       0.036            0.009             0.050   
Political Interest          0.057            0.015             0.034   
Church                      0.035            0.013             0.031   
Gender                      0.037            0.013             0.035   
Age                         0.037            0.014             0.033   
Race                        0.037            0.028             0.028   
Discuss Politics            0.042            0.005             0.032   

condition                preamble                                     
item               Race Diversity Refugee Allowing Income Inequality  
Group                                                                 
Ideology                   -0.017            0.133             0.065  
Party                      -0.009            0.118             0.057  
Political Interest          0.005            0.157             0.048  
Church                     -0.013            0.133             0.035  
Gender                     -0.013            0.134             0.038  
Age                        -0.012            0.134             0.036  
Race                       -0.001            0.145             0.021  
Discuss Politics           -0.033            0.147             0.013


delta |D|  (negative = improvement)


condition            reformulated                                     \
item               Race Diversity Refugee Allowing Income Inequality   
Group                                                                  
Ideology                   -0.172           -0.122            -0.040   
Party                      -0.147           -0.117            -0.016   
Political Interest         -0.153           -0.169            -0.070   
Church                     -0.159           -0.152            -0.044   
Gender                     -0.159           -0.151            -0.041   
Age                        -0.158           -0.153            -0.042   
Race                       -0.124           -0.151            -0.043   
Discuss Politics           -0.166           -0.212            -0.078   

condition                reversed                                     \
item               Race Diversity Refugee Allowing Income Inequality   
Group                                                                  
Ideology                   -0.146           -0.086             0.145   
Party                      -0.132           -0.109             0.078   
Political Interest         -0.145           -0.124            -0.200   
Church                     -0.137           -0.085            -0.146   
Gender                     -0.140           -0.085            -0.165   
Age                        -0.139           -0.086            -0.150   
Race                       -0.142           -0.108            -0.190   
Discuss Politics           -0.185           -0.111            -0.276   

condition                 priming                                     \
item               Race Diversity Refugee Allowing Income Inequality   
Group                                                                  
Ideology                    0.023            0.027             0.092   
Party                       0.013            0.026             0.078   
Political Interest          0.024            0.032             0.068   
Church                      0.013            0.033             0.072   
Gender                      0.013            0.033             0.077   
Age                         0.013            0.035             0.075   
Race                        0.014            0.041             0.067   
Discuss Politics            0.014            0.018             0.057   

condition                preamble                                     
item               Race Diversity Refugee Allowing Income Inequality  
Group                                                                 
Ideology                   -0.011            0.145             0.085  
Party                      -0.005            0.125             0.080  
Political Interest          0.005            0.156             0.080  
Church                     -0.005            0.142             0.069  
Gender                     -0.006            0.143             0.072  
Age                        -0.006            0.144             0.073  
Race                       -0.000            0.133             0.043  
Discuss Politics           -0.018            0.124             0.024

### Per subgroup level (no averaging)

In [15]:
# substantive level ordering, taken from `groups` so it cannot drift
_CAT_ORDER = {group_var_dict[k]: list(v.values()) for k, v in groups.items()}
_CAT_ORDER["Age"] = list(AGE_LABELS.values())

MIN_N = 30      # mask a cell when the model or human subgroup has fewer responses

_lv = _sub.copy()
_thin = (_lv["n_model"] < MIN_N) | (_lv["n_human"] < MIN_N)
if _thin.any():
    print(f"masked {_thin.sum()} cells below MIN_N={MIN_N}:")
    for _, r in _lv[_thin].drop_duplicates(["Group", "Category", "item"]).iterrows():
        print(f"  {r['Group']}/{r['Category']}/{r['item']}: "
              f"n_model={int(r['n_model'])}, n_human={int(r['n_human'])}")
    _lv.loc[_thin, ["delta_jsd", "delta_absD", "D_base"]] = np.nan

_lidx = pd.MultiIndex.from_tuples(
    [(g, c) for g in _gord for c in _CAT_ORDER[g]], names=["Group", "Category"])

def by_level(metric, nd=3):
    return (_lv.pivot_table(index=["Group", "Category"], columns=["condition", "item"],
                            values=metric)
            .reindex(index=_lidx).reindex(columns=_cols).round(nd))

def base_D_level(nd=3):
    b = (_lv[_lv["condition"] == CONDITIONS[0]]
         .pivot_table(index=["Group", "Category"], columns="item", values="D_base"))
    return b.reindex(index=_lidx)[ITEMS].round(nd)

pd.set_option("display.max_rows", 200)

print("Baseline signed desirability gap D at Replicate, per level")
display(base_D_level())

print("\ndelta JSD per level")
display(by_level("delta_jsd"))

print("\ndelta |D| per level")
display(by_level("delta_absD"))

Baseline signed desirability gap D at Replicate, per level


item                                          Race Diversity  \
Group              Category                                    
Ideology           Extremely liberal                   0.164   
                   Liberal                             0.154   
                   Slightly liberal                    0.223   
                   Moderate                            0.445   
                   Slightly conservative               0.524   
                   Conservative                        0.627   
                   Extremely conservative              0.577   
Party              Strong democrat                     0.240   
                   Weak democrat                       0.341   
                   Democrat-leaning indep.             0.240   
                   Independent                         0.542   
                   Republican-leaning indep.           0.520   
                   Weak republican                     0.550   
                   Strong republican                   0.641   
Political Interest Very                                0.334   
                   Somewhat                            0.430   
                   Not very                            0.480   
                   Not at all                          0.575   
Church             Attend church                       0.481   
                   Does not attend church              0.381   
Gender             Man                                 0.438   
                   Woman                               0.423   
Age                18-29                               0.424   
                   30-39                               0.381   
                   40-49                               0.425   
                   50-59                               0.433   
                   60-69                               0.424   
                   70+                                 0.474   
Race               White                               0.433   
                   Black                               0.456   
                   Asian                               0.422   
                   Native American                     0.340   
                   Hispanic                            0.430   
Discuss Politics   Like to discuss politics            0.415   
                   Never discuss politics              0.600   

item                                          Refugee Allowing  \
Group              Category                                      
Ideology           Extremely liberal                     0.089   
                   Liberal                               0.110   
                   Slightly liberal                      0.145   
                   Moderate                              0.383   
                   Slightly conservative                 0.456   
                   Conservative                          0.304   
                   Extremely conservative                0.151   
Party              Strong democrat                       0.232   
                   Weak democrat                         0.294   
                   Democrat-leaning indep.               0.239   
                   Independent                           0.476   
                   Republican-leaning indep.             0.454   
                   Weak republican                       0.416   
                   Strong republican                     0.120   
Political Interest Very                                  0.204   
                   Somewhat                              0.278   
                   Not very                              0.368   
                   Not at all                            0.338   
Church             Attend church                         0.347   
                   Does not attend church                0.232   
Gender             Man                                   0.288   
                   Woman                                 0.284   
Age                18-29                       


delta JSD per level


condition                                      reformulated                   \
item                                         Race Diversity Refugee Allowing   
Group              Category                                                    
Ideology           Extremely liberal                 -0.021           -0.011   
                   Liberal                           -0.016           -0.029   
                   Slightly liberal                  -0.046           -0.010   
                   Moderate                          -0.064           -0.096   
                   Slightly conservative             -0.125           -0.103   
                   Conservative                      -0.154           -0.063   
                   Extremely conservative             0.004           -0.010   
Party              Strong democrat                   -0.004           -0.038   
                   Weak democrat                     -0.034           -0.043   
                   Democrat-leaning indep.            0.000           -0.052   
                   Independent                       -0.142           -0.159   
                   Republican-leaning indep.         -0.096           -0.152   
                   Weak republican                   -0.129           -0.099   
                   Strong republican                 -0.115            0.029   
Political Interest Very                              -0.043           -0.021   
                   Somewhat                          -0.067           -0.040   
                   Not very                          -0.082           -0.100   
                   Not at all                        -0.146           -0.047   
Church             Attend church                     -0.102           -0.078   
                   Does not attend church            -0.052           -0.049   
Gender             Man                               -0.089           -0.057   
                   Woman                             -0.061           -0.066   
Age                18-29                             -0.072           -0.048   
                   30-39                             -0.034           -0.055   
                   40-49                             -0.068           -0.062   
                   50-59                             -0.083           -0.062   
                   60-69                             -0.091           -0.073   
                   70+                               -0.086           -0.055   
Race               White                             -0.082           -0.060   
                   Black                             -0.083           -0.053   
                   Asian                             -0.007           -0.072   
                   Native American                   -0.005           -0.053   
                   Hispanic                          -0.006           -0.105   
Discuss Politics   Like to discuss politics          -0.059           -0.053   
                   Never discuss politics            -0.201           -0.190   

condition                                                            reversed  \
item                                         Income Inequality Race Diversity   
Group              Category                                                     
Ideology           Extremely liberal                     0.037         -0.019   
                   Liberal                               0.049         -0.021   
                   Slightly liberal                      0.064         -0.072   
                   Moderate                              0.027         -0.171   
                   Slightly conservative                 0.001         -0.185   
                   Conservative                          0.051         -0.278   
                   Extremely conservative                0.057         -0.230   
Party              Strong democrat                       0.031         -0.059   
                   Weak democrat                         0.042      


delta |D| per level


condition                                      reformulated                   \
item                                         Race Diversity Refugee Allowing   
Group              Category                                                    
Ideology           Extremely liberal                 -0.065           -0.074   
                   Liberal                           -0.019           -0.077   
                   Slightly liberal                  -0.059           -0.104   
                   Moderate                          -0.034           -0.182   
                   Slightly conservative             -0.240           -0.171   
                   Conservative                      -0.393           -0.178   
                   Extremely conservative            -0.392           -0.064   
Party              Strong democrat                   -0.002           -0.028   
                   Weak democrat                     -0.017           -0.067   
                   Democrat-leaning indep.            0.000           -0.041   
                   Independent                       -0.109           -0.235   
                   Republican-leaning indep.         -0.206           -0.302   
                   Weak republican                   -0.222           -0.175   
                   Strong republican                 -0.476            0.031   
Political Interest Very                              -0.175           -0.102   
                   Somewhat                          -0.157           -0.095   
                   Not very                          -0.124           -0.238   
                   Not at all                        -0.158           -0.243   
Church             Attend church                     -0.175           -0.160   
                   Does not attend church            -0.143           -0.144   
Gender             Man                               -0.161           -0.134   
                   Woman                             -0.157           -0.167   
Age                18-29                             -0.114           -0.190   
                   30-39                             -0.130           -0.151   
                   40-49                             -0.161           -0.172   
                   50-59                             -0.177           -0.130   
                   60-69                             -0.189           -0.157   
                   70+                               -0.175           -0.120   
Race               White                             -0.173           -0.154   
                   Black                             -0.131           -0.118   
                   Asian                             -0.107           -0.166   
                   Native American                   -0.134           -0.124   
                   Hispanic                          -0.075           -0.195   
Discuss Politics   Like to discuss politics          -0.152           -0.138   
                   Never discuss politics            -0.180           -0.287   

condition                                                            reversed  \
item                                         Income Inequality Race Diversity   
Group              Category                                                     
Ideology           Extremely liberal                    -0.031         -0.008   
                   Liberal                              -0.049         -0.013   
                   Slightly liberal                     -0.068         -0.050   
                   Moderate                             -0.013         -0.145   
                   Slightly conservative                -0.152         -0.128   
                   Conservative                         -0.014         -0.276   
                   Extremely conservative                0.047         -0.404   
Party              Strong democrat                       0.021         -0.041   
                   Weak democrat                        -0.016      

### Save

In [ ]:
import os
outdir = "../tables-stochastic/stratified"
os.makedirs(outdir, exist_ok=True)

cols = ["condition", "Group", "Category", "item", "D_base", "D_cond",
        "delta_jsd", "delta_absD", "jsd_base", "jsd_cond", "n_model", "n_human"]
strat[cols].round(4).to_csv(f"{outdir}/{model_name}_stratified_long.csv", index=False)

for name, tbl in [("baseD_group", base_D()), ("deltaJSD_group", by_group("delta_jsd")),
                  ("deltaabsD_group", by_group("delta_absD")),
                  ("baseD_level", base_D_level()), ("deltaJSD_level", by_level("delta_jsd")),
                  ("deltaabsD_level", by_level("delta_absD"))]:
    tbl.to_csv(f"{outdir}/{model_name}_{name}.csv")

print("saved to", outdir)

## Full results: all items, all four conditions

In [ ]:
ALL_ITEMS = [name_dict[q] for q in qids if name_dict[q] in set(strat["item"])]
_dropped  = [name_dict[q] for q in qids if name_dict[q] not in set(strat["item"])]
print(f"{len(ALL_ITEMS)} directional items:", ALL_ITEMS)
if _dropped:
    print("absent (non-directional, no desirability gap):", _dropped)

# coverage per condition: the reverse-coded run carries only its own subset
_cov = (strat.groupby("condition")["item"].nunique().reindex(CONDITIONS))
print("\nitems present per condition:")
for c, n in _cov.items():
    miss = sorted(set(ALL_ITEMS) - set(strat.loc[strat.condition == c, "item"]))
    print(f"  {c:14s} {n}" + (f"   missing: {miss}" if miss else ""))

COMMON_ITEMS = sorted(set.intersection(*(set(strat.loc[strat.condition == c, "item"])
                                         for c in CONDITIONS)), key=ALL_ITEMS.index)
print("\nitems common to all four conditions:", COMMON_ITEMS)
print("NOTE: the Average column below spans whatever items a condition has, so the reverse-coded")
print("      average covers fewer items. Pass items=COMMON_ITEMS for a comparable average.")

MIN_N_FULL = 30            # mask a subgroup cell below this many responses

_full = strat.copy()
_thin = (_full["n_model"] < MIN_N_FULL) | (_full["n_human"] < MIN_N_FULL)
if _thin.any():
    print(f"\nmasked {_thin.sum()} thin cells (< {MIN_N_FULL} responses):")
    for _, r in (_full[_thin].drop_duplicates(["Group", "Category"])
                 .sort_values(["Group", "Category"]).iterrows()):
        print(f"  {r['Group']}/{r['Category']}: n_model={int(r['n_model'])}, "
              f"n_human={int(r['n_human'])}")
    _full.loc[_thin, ["delta_jsd", "delta_absD", "D_base"]] = np.nan

_gord_full = [g for g in GROUP_ORDER if g in set(_full["Group"])]
_lidx_full = pd.MultiIndex.from_tuples(
    [(g, c) for g in _gord_full for c in _CAT_ORDER[g]], names=["Group", "Category"])


def full_group(metric, condition, items=None, nd=3, average=True):
    """Subgroup-averaged metric for one condition: variables down, items across."""
    items = items or ALL_ITEMS
    t = (_full[_full["condition"] == condition]
         .pivot_table(index="Group", columns="item", values=metric, aggfunc="mean")
         .reindex(index=_gord_full).reindex(columns=items))
    if average:
        t["Average"] = t.mean(axis=1)
    return t.round(nd)


def full_level(metric, condition, items=None, nd=3, average=True):
    """Same, without averaging over subgroups: every level down, items across."""
    items = items or ALL_ITEMS
    t = (_full[_full["condition"] == condition]
         .pivot_table(index=["Group", "Category"], columns="item", values=metric)
         .reindex(index=_lidx_full).reindex(columns=items))
    if average:
        t["Average"] = t.mean(axis=1)
    return t.round(nd)


def full_baseD(level=False, items=None, nd=3):
    """Baseline signed gap D at Replicate. Identical across conditions, so read one block."""
    items = items or ALL_ITEMS
    idx = ["Group", "Category"] if level else "Group"
    t = (_full[_full["condition"] == CONDITIONS[0]]
         .pivot_table(index=idx, columns="item", values="D_base"))
    return t.reindex(index=_lidx_full if level else _gord_full).reindex(columns=items).round(nd)

9 directional items: ['Gay Marriage', 'Refugee Allowing', 'Income Inequality', 'Gender Role', 'Climate Change', 'Gun Regulation', 'Drug Addiction', 'Race Diversity', 'Health Insurance']
absent (non-directional, no desirability gap): ['Current Economy']

items present per condition:
  reformulated   9
  reversed       6   missing: ['Drug Addiction', 'Gun Regulation', 'Health Insurance']
  priming        9
  preamble       9

items common to all four conditions: ['Gay Marriage', 'Refugee Allowing', 'Income Inequality', 'Gender Role', 'Climate Change', 'Race Diversity']
NOTE: the Average column below spans whatever items a condition has, so the reverse-coded
      average covers fewer items. Pass items=COMMON_ITEMS for a comparable average.


In [ ]:
# Group level (averaged within each demographic variable)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 250)

print("Baseline signed desirability gap D at Replicate\n")
display(full_baseD())

for metric, title in [("delta_jsd", "delta JSD"), ("delta_absD", "delta |D|")]:
    for c in CONDITIONS:
        print(f"\n{title}  |  {c}   (negative = improvement)")
        display(full_group(metric, c))

Baseline signed desirability gap D at Replicate



item,Gay Marriage,Refugee Allowing,Income Inequality,Gender Role,Climate Change,Gun Regulation,Drug Addiction,Race Diversity,Health Insurance
Group,,,,,,,,,
Ideology,0.135,0.234,0.190,-0.041,0.098,0.040,0.282,0.388,0.189
Party,0.156,0.319,0.242,-0.048,0.100,0.052,0.316,0.439,0.221
Political Interest,0.147,0.297,0.245,-0.051,0.097,0.057,0.311,0.455,0.213
Church,0.155,0.289,0.228,-0.047,0.096,0.050,0.302,0.431,0.201
Gender,0.150,0.286,0.220,-0.048,0.095,0.051,0.301,0.430,0.197
Age,0.145,0.279,0.217,-0.051,0.092,0.047,0.302,0.427,0.190
Race,0.164,0.322,0.214,-0.071,0.080,0.041,0.292,0.416,0.203
Discuss Politics,0.179,0.398,0.325,-0.055,0.126,0.088,0.324,0.508,0.265



delta JSD  |  reformulated   (negative = improvement)


item,Gay Marriage,Refugee Allowing,Income Inequality,Gender Role,Climate Change,Gun Regulation,Drug Addiction,Race Diversity,Health Insurance,Average
Group,,,,,,,,,,
Ideology,-0.004,-0.046,0.041,-0.013,-0.040,0.037,-0.013,-0.060,0.034,-0.007
Party,0.003,-0.073,0.035,-0.059,-0.032,0.025,-0.013,-0.074,0.025,-0.018
Political Interest,-0.007,-0.052,0.034,-0.065,-0.022,0.009,-0.025,-0.085,0.019,-0.021
Church,-0.006,-0.064,0.045,-0.060,-0.032,0.013,-0.024,-0.077,0.023,-0.020
Gender,-0.006,-0.061,0.047,-0.059,-0.034,0.013,-0.023,-0.075,0.023,-0.019
Age,-0.008,-0.059,0.050,-0.058,-0.031,0.012,-0.023,-0.072,0.023,-0.019
Race,-0.008,-0.069,0.069,-0.070,-0.022,0.009,-0.023,-0.036,0.013,-0.015
Discuss Politics,-0.019,-0.122,0.036,-0.084,-0.040,-0.001,-0.023,-0.130,0.006,-0.042



delta JSD  |  reversed   (negative = improvement)


...

item,Gay Marriage,Refugee Allowing,Income Inequality,Gender Role,Climate Change,Gun Regulation,Drug Addiction,Race Diversity,Health Insurance,Average
Group,,,,,,,,,,
Ideology,-0.036,-0.086,0.145,0.900,0.181,NaN,NaN,-0.146,NaN,0.160
Party,-0.021,-0.109,0.078,0.900,0.211,NaN,NaN,-0.132,NaN,0.154
Political Interest,-0.030,-0.124,-0.200,0.892,0.215,NaN,NaN,-0.145,NaN,0.101
Church,-0.026,-0.085,-0.146,0.899,0.205,NaN,NaN,-0.137,NaN,0.118
Gender,-0.026,-0.085,-0.165,0.898,0.206,NaN,NaN,-0.140,NaN,0.115
Age,-0.031,-0.086,-0.150,0.893,0.204,NaN,NaN,-0.139,NaN,0.115
Race,-0.013,-0.108,-0.190,0.851,0.195,NaN,NaN,-0.142,NaN,0.099
Discuss Politics,-0.022,-0.111,-0.276,0.886,0.211,NaN,NaN,-0.185,NaN,0.084



delta |D|  |  priming   (negative = improvement)


item,Gay Marriage,Refugee Allowing,Income Inequality,Gender Role,Climate Change,Gun Regulation,Drug Addiction,Race Diversity,Health Insurance,Average
Group,,,,,,,,,,
Ideology,0.006,0.027,0.092,0.013,0.143,-0.001,0.012,0.023,0.082,0.044
Party,0.010,0.026,0.078,0.005,0.134,0.001,0.004,0.013,0.075,0.038
Political Interest,0.012,0.032,0.068,0.005,0.151,0.030,0.003,0.024,0.070,0.044
Church,-0.002,0.033,0.072,0.006,0.143,0.029,0.004,0.013,0.076,0.042
Gender,0.000,0.033,0.077,0.006,0.143,0.027,0.005,0.013,0.077,0.042
Age,0.001,0.035,0.075,0.006,0.141,0.019,0.005,0.013,0.078,0.041
Race,0.013,0.041,0.067,0.006,0.125,-0.008,0.003,0.014,0.046,0.034
Discuss Politics,0.006,0.018,0.057,0.004,0.137,0.015,0.005,0.014,0.039,0.033



delta |D|  |  preamble   (negative = improvement)


item,Gay Marriage,Refugee Allowing,Income Inequality,Gender Role,Climate Change,Gun Regulation,Drug Addiction,Race Diversity,Health Insurance,Average
Group,,,,,,,,,,
Ideology,0.051,0.145,0.085,0.009,0.053,0.044,0.012,-0.011,0.188,0.064
Party,0.048,0.125,0.080,0.003,0.060,0.032,0.004,-0.005,0.185,0.059
Political Interest,0.056,0.156,0.080,0.003,0.062,0.089,0.003,0.005,0.203,0.073
Church,0.049,0.142,0.069,0.004,0.064,0.070,0.004,-0.005,0.185,0.065
Gender,0.048,0.143,0.072,0.004,0.063,0.070,0.005,-0.006,0.186,0.065
Age,0.047,0.144,0.073,0.004,0.062,0.060,0.005,-0.006,0.190,0.064
Race,0.034,0.133,0.043,0.002,0.050,0.012,0.003,-0.000,0.163,0.049
Discuss Politics,0.038,0.124,0.024,0.003,0.064,0.061,0.005,-0.018,0.157,0.051


In [ ]:
# Subgroup level (no averaging)
pd.set_option("display.max_rows", 300)

print("Baseline signed desirability gap D at Replicate, per subgroup\n")
display(full_baseD(level=True))

for metric, title in [("delta_jsd", "delta JSD"), ("delta_absD", "delta |D|")]:
    for c in CONDITIONS:
        print(f"\n{title}  |  {c}   (negative = improvement)")
        display(full_level(metric, c))

Baseline signed desirability gap D at Replicate, per subgroup



item                                          Gay Marriage  Refugee Allowing  Income Inequality  Gender Role  Climate Change  Gun Regulation  Drug Addiction  Race Diversity  Health Insurance
Group              Category                                                                                                                                                                   
Ideology           Extremely liberal                 0.083             0.089              0.083        0.007           0.061           0.022           0.146           0.164             0.056
                   Liberal                           0.046             0.110              0.075       -0.082           0.017          -0.086           0.173           0.154             0.080
                   Slightly liberal                  0.073             0.145              0.125       -0.067          -0.031          -0.062           0.273           0.223             0.087
                   Moderate                          0.168             0.383              0.279       -0.044           0.043           0.056           0.323           0.445             0.253
                   Slightly conservative             0.169             0.456              0.382       -0.040           0.182           0.114           0.358           0.524             0.350
                   Conservative                      0.193             0.304              0.230       -0.021           0.209           0.120           0.401           0.627             0.288
                   Extremely conservative            0.212             0.151              0.154       -0.043           0.204           0.117           0.301           0.577             0.208
Party              Strong democrat                   0.145             0.232              0.234       -0.057           0.036           0.121           0.178           0.240             0.164
                   Weak democrat                     0.103             0.294              0.198       -0.067          -0.014           0.051           0.258           0.341             0.144
                   Democrat-leaning indep.           0.125             0.239              0.243       -0.063           0.026           0.197           0.231           0.240             0.231
                   Independent                       0.168             0.476              0.330       -0.060           0.141           0.153           0.399           0.542             0.308
                   Republican-leaning indep.         0.258             0.454              0.280       -0.035           0.206          -0.066           0.399           0.520             0.300
                   Weak republican                   0.168             0.416              0.401       -0.027           0.154           0.060           0.378           0.550             0.339
                   Strong republican                 0.123             0.120              0.008       -0.028           0.153          -0.154           0.365           0.641             0.058
Political Interest Very                              0.122             0.204              0.144       -0.051           0.134           0.029           0.238           0.334             0.134
                   Somewhat                          0.168             0.278              0.220       -0.048           0.072           0.046           0.303           0.430             0.196
                   Not very                          0.167             0.368              0.306       -0.046           0.062           0.067           0.356           0.480             0.243
                   Not at all                        0.129             0.338              0.312       -0.060           0.119           0.087           0.348           0.575             0.279
Church             Attend church                     0.233             0.347              0.317       -0.040           0.142           0.092           0.319           0.481        


delta JSD  |  reformulated   (negative = improvement)


item                                          Gay Marriage  Refugee Allowing  Income Inequality  Gender Role  Climate Change  Gun Regulation  Drug Addiction  Race Diversity  Health Insurance  Average
Group              Category                                                                                                                                                                            
Ideology           Extremely liberal                -0.016            -0.011              0.037        0.251          -0.005           0.020           0.000          -0.021             0.014    0.030
                   Liberal                           0.000            -0.029              0.049        0.000           0.026           0.027          -0.005          -0.016             0.014    0.007
                   Slightly liberal                 -0.005            -0.010              0.064       -0.062           0.060           0.028          -0.015          -0.046             0.013    0.003
                   Moderate                         -0.022            -0.096              0.027       -0.098          -0.013           0.011          -0.015          -0.064             0.032   -0.026
                   Slightly conservative            -0.012            -0.103              0.001       -0.082          -0.109           0.005          -0.040          -0.125             0.058   -0.045
                   Conservative                      0.007            -0.063              0.051       -0.070          -0.099           0.027          -0.040          -0.154             0.016   -0.036
                   Extremely conservative            0.024            -0.010              0.057       -0.032          -0.139           0.139           0.022           0.004             0.093    0.018
Party              Strong democrat                   0.007            -0.038              0.031       -0.013          -0.009           0.035           0.000          -0.004             0.021    0.003
                   Weak democrat                     0.019            -0.043              0.042       -0.065           0.049          -0.006          -0.004          -0.034             0.036   -0.001
                   Democrat-leaning indep.          -0.005            -0.052              0.031       -0.039           0.011          -0.032           0.000           0.000             0.021   -0.007
                   Independent                       0.014            -0.159              0.046       -0.101          -0.079          -0.007          -0.032          -0.142             0.001   -0.051
                   Republican-leaning indep.        -0.005            -0.152             -0.054       -0.135          -0.123           0.019          -0.040          -0.096             0.032   -0.062
                   Weak republican                   0.013            -0.099              0.002       -0.060          -0.076          -0.007          -0.019          -0.129             0.015   -0.040
                   Strong republican                -0.024             0.029              0.145       -0.001           0.005           0.174           0.001          -0.115             0.045    0.029
Political Interest Very                             -0.008            -0.021              0.045       -0.043          -0.048           0.035          -0.004          -0.043             0.026   -0.007
                   Somewhat                         -0.011            -0.040              0.058       -0.072          -0.041           0.018          -0.021          -0.067             0.030   -0.016
                   Not very                         -0.011            -0.100              0.022       -0.084          -0.015          -0.009          -0.030          -0.082            -0.001   -0.035
                   Not at all                        0.003            -0.047              0.011       -0.063           0.018          -0.006          -0.044          -0.146             0.020   -0.028



delta JSD  |  reversed   (negative = improvement)


...

item                                          Gay Marriage  Refugee Allowing  Income Inequality  Gender Role  Climate Change  Gun Regulation  Drug Addiction  Race Diversity  Health Insurance  Average
Group              Category                                                                                                                                                                            
Ideology           Extremely liberal                 0.000            -0.027              0.208        0.903           0.038             NaN             NaN          -0.008               NaN    0.186
                   Liberal                          -0.003            -0.039              0.175        0.830           0.087             NaN             NaN          -0.013               NaN    0.173
                   Slightly liberal                 -0.028            -0.081              0.048        0.862           0.124             NaN             NaN          -0.050               NaN    0.146
                   Moderate                         -0.029            -0.155             -0.122        0.912           0.243             NaN             NaN          -0.145               NaN    0.117
                   Slightly conservative             0.073            -0.212             -0.085        0.921           0.284             NaN             NaN          -0.128               NaN    0.142
                   Conservative                     -0.059            -0.041              0.305        0.958           0.319             NaN             NaN          -0.276               NaN    0.201
                   Extremely conservative           -0.210            -0.049              0.486        0.914           0.173             NaN             NaN          -0.404               NaN    0.152
Party              Strong democrat                  -0.029            -0.026              0.047        0.871           0.085             NaN             NaN          -0.041               NaN    0.151
                   Weak democrat                    -0.052            -0.116              0.077        0.858           0.159             NaN             NaN          -0.068               NaN    0.143
                   Democrat-leaning indep.          -0.011            -0.054              0.141        0.872           0.134             NaN             NaN          -0.047               NaN    0.173
                   Independent                      -0.023            -0.246             -0.309        0.879           0.199             NaN             NaN          -0.103               NaN    0.066
                   Republican-leaning indep.        -0.004            -0.196              0.081        0.931           0.300             NaN             NaN          -0.188               NaN    0.154
                   Weak republican                   0.003            -0.161             -0.205        0.944           0.289             NaN             NaN          -0.156               NaN    0.119
                   Strong republican                -0.029             0.038              0.714        0.942           0.309             NaN             NaN          -0.320               NaN    0.276
Political Interest Very                             -0.022            -0.024             -0.125        0.888           0.130             NaN             NaN          -0.083               NaN    0.127
                   Somewhat                         -0.026            -0.089             -0.123        0.902           0.230             NaN             NaN          -0.129               NaN    0.127
                   Not very                         -0.029            -0.166             -0.289        0.907           0.277             NaN             NaN          -0.192               NaN    0.085
                   Not at all                       -0.044            -0.218             -0.264        0.869           0.222             NaN             NaN          -0.173               NaN    0.065



delta |D|  |  priming   (negative = improvement)


item                                          Gay Marriage  Refugee Allowing  Income Inequality  Gender Role  Climate Change  Gun Regulation  Drug Addiction  Race Diversity  Health Insurance  Average
Group              Category                                                                                                                                                                            
Ideology           Extremely liberal                 0.000             0.011              0.057        0.082           0.035           0.096           0.000           0.000             0.043    0.036
                   Liberal                           0.004             0.014              0.089        0.006           0.066          -0.056           0.000           0.003             0.038    0.018
                   Slightly liberal                 -0.018             0.054              0.143        0.003           0.051          -0.029           0.000           0.000             0.106    0.035
                   Moderate                         -0.033             0.024              0.088        0.000           0.125           0.027           0.000           0.004             0.063    0.033
                   Slightly conservative             0.040             0.044              0.134        0.000           0.154           0.036           0.000           0.002             0.115    0.058
                   Conservative                      0.024             0.082              0.091        0.000           0.251          -0.008           0.001           0.023             0.064    0.059
                   Extremely conservative            0.025            -0.042              0.043        0.000           0.316          -0.074           0.080           0.128             0.141    0.069
Party              Strong democrat                   0.002            -0.002              0.011        0.016           0.072          -0.011           0.000           0.002             0.010    0.011
                   Weak democrat                     0.025             0.016              0.065        0.009           0.084           0.061           0.000           0.005             0.077    0.038
                   Democrat-leaning indep.           0.000            -0.005              0.015        0.002           0.055          -0.042           0.000           0.000             0.004    0.003
                   Independent                       0.030            -0.018              0.028        0.001           0.113           0.076           0.000           0.008             0.052    0.032
                   Republican-leaning indep.         0.012             0.021              0.152        0.000           0.162          -0.050           0.000           0.017             0.132    0.050
                   Weak republican                   0.023             0.032              0.046        0.002           0.168           0.061           0.009           0.020             0.036    0.044
                   Strong republican                -0.019             0.138              0.231        0.002           0.285          -0.088           0.019           0.040             0.212    0.091
Political Interest Very                             -0.024             0.021              0.096        0.009           0.099           0.048           0.008           0.002             0.087    0.039
                   Somewhat                         -0.012             0.049              0.081        0.002           0.148           0.020           0.005           0.009             0.062    0.040
                   Not very                          0.013             0.042              0.053        0.001           0.172           0.031           0.001           0.029             0.083    0.047
                   Not at all                        0.071             0.017              0.041        0.008           0.183           0.022           0.000           0.058             0.047    0.050



delta |D|  |  preamble   (negative = improvement)


item                                          Gay Marriage  Refugee Allowing  Income Inequality  Gender Role  Climate Change  Gun Regulation  Drug Addiction  Race Diversity  Health Insurance  Average
Group              Category                                                                                                                                                                            
Ideology           Extremely liberal                 0.000             0.023              0.058        0.056           0.031           0.127           0.000           0.000             0.047    0.038
                   Liberal                           0.001             0.023              0.076        0.005           0.048          -0.035           0.000           0.001             0.053    0.019
                   Slightly liberal                  0.013             0.098              0.086        0.003          -0.001          -0.045           0.000          -0.003             0.135    0.032
                   Moderate                          0.028             0.101              0.085        0.000           0.066           0.053           0.000          -0.001             0.151    0.054
                   Slightly conservative             0.084             0.130              0.098       -0.002           0.082           0.070           0.000          -0.004             0.245    0.078
                   Conservative                      0.102             0.304              0.086        0.000           0.082           0.053           0.001          -0.020             0.342    0.106
                   Extremely conservative            0.127             0.339              0.106        0.000           0.063           0.088           0.080          -0.052             0.345    0.122
Party              Strong democrat                   0.004             0.005              0.014        0.014           0.042           0.022           0.000           0.002             0.012    0.013
                   Weak democrat                     0.038             0.053              0.066        0.005           0.023           0.097           0.000          -0.001             0.118    0.044
                   Democrat-leaning indep.           0.000             0.005              0.030        0.002           0.042           0.022           0.000           0.000             0.008    0.012
                   Independent                       0.061             0.068              0.057       -0.001           0.040           0.079           0.000          -0.002             0.137    0.049
                   Republican-leaning indep.         0.038             0.106              0.102        0.000           0.076           0.000           0.000          -0.008             0.314    0.070
                   Weak republican                   0.076             0.152              0.097        0.002           0.089           0.113           0.009          -0.009             0.226    0.084
                   Strong republican                 0.117             0.486              0.192        0.001           0.106          -0.111           0.019          -0.014             0.477    0.141
Political Interest Very                              0.048             0.117              0.078        0.006           0.056           0.080           0.008          -0.007             0.179    0.063
                   Somewhat                          0.033             0.148              0.059        0.000           0.072           0.044           0.005          -0.016             0.166    0.057
                   Not very                          0.043             0.156              0.072        0.000           0.066           0.095           0.001           0.011             0.232    0.075
                   Not at all                        0.099             0.202              0.111        0.008           0.056           0.137           0.000           0.029             0.235    0.097


In [29]:
# Baseline sign audit, all variables including Age
_b = _full[_full["condition"] == CONDITIONS[0]].dropna(subset=["D_base"])
_neg = _b[_b["D_base"] < 0]

print(f"{len(_neg)} of {len(_b)} subgroup-item cells have D < 0")
print(f"{_b['Category'].nunique()} subgroups, {_b['item'].nunique()} items")
print("\nnegative cells by item (out of subgroups with that item):")
_by = (_neg.groupby("item")["Category"].nunique()
       .to_frame("n_negative")
       .join(_b.groupby("item")["Category"].nunique().rename("n_subgroups"))
       .sort_values("n_negative", ascending=False))
display(_by)

print("\nAge bins with D < 0:")
display(_neg[_neg["Group"] == "Age"][["Category", "item", "D_base"]]
        .sort_values(["item", "Category"]).round(3))

43 of 315 subgroup-item cells have D < 0
35 subgroups, 9 items

negative cells by item (out of subgroups with that item):


,n_negative,n_subgroups
item,,
Gender Role,34,35
Gun Regulation,7,35
Climate Change,2,35



Age bins with D < 0:


,Category,item,D_base
264,18-29,Gender Role,-0.096
273,30-39,Gender Role,-0.073
282,40-49,Gender Role,-0.060
291,50-59,Gender Role,-0.033
300,60-69,Gender Role,-0.022
309,70+,Gender Role,-0.021
266,18-29,Gun Regulation,-0.023


In [30]:
_b = _full[_full["condition"] == CONDITIONS[0]]
_r = 1 - _b["n_human"] / _b["n_model"]
print(f"human cells smaller by: mean {_r.mean():.1%}, median {_r.median():.1%}, "
      f"range {_r.min():.1%} to {_r.max():.1%}")

# per item, since non-response varies by question
display((1 - _b.groupby("item")["n_human"].sum() / _b.groupby("item")["n_model"].sum())
        .mul(100).round(1).sort_values().to_frame("% smaller"))

human cells smaller by: mean 10.0%, median 12.1%, range -7.4% to 24.4%


,% smaller
item,
Gay Marriage,1.2
Gun Regulation,9.3
Refugee Allowing,9.7
Climate Change,9.8
Income Inequality,10.0
Gender Role,10.4
Race Diversity,10.4
Drug Addiction,10.5
Health Insurance,10.7
